# Week 10 · Day 2 — Comparing CNN Architectures on a Real Task (Skin Cancer)

**You're in the field now.** Today we classify **skin lesions** from dermatoscopy images — the same kind of task real dermatology-AI systems do — and we put the famous CNN architectures head-to-head to see which is best *for this job*.

- Dataset: **HAM10000** — 10,015 dermatoscopy images, **7 lesion types** (including melanoma, the dangerous one).
- We compare **5 architectures** you met in the CNN-history talk: **InceptionV3, ResNet50, DenseNet121, MobileNetV2, EfficientNetB0**.
- Method: **feature extraction** — freeze each pretrained backbone, train a small head, benchmark it.
- We compare on three real-world axes: **accuracy · speed · model size**.
- Then we take the winner and **fine-tune** it for a final boost.

> **Kaggle GPU:** Settings → Accelerator → GPU, then add the HAM10000 dataset via Add Input.

> ⚠️ **Real-world catch:** this dataset is badly **imbalanced** (one class is ~67% of it). Plain accuracy lies here — we'll use metrics that don't.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
import time

tf.random.set_seed(42)
print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Load HAM10000 (CSV-driven — a real dataset layout)

- Real datasets often aren't tidy class folders. HAM10000 gives you:
  - a **metadata CSV** mapping each `image_id` → diagnosis (`dx`),
  - images split across **two folders** (`part_1`, `part_2`).
- So we read the CSV, then build each image's full path with `os`. This is the realistic version of what we've been doing.

In [ ]:
# Kaggle path for the classic HAM10000 dataset — adjust if yours differs
BASE = "/kaggle/input/skin-cancer-mnist-ham10000"

meta = pd.read_csv(os.path.join(BASE, "HAM10000_metadata.csv"))
print("rows:", len(meta))
print(meta[["image_id", "dx"]].head())

# the 7 lesion classes and what they mean
DX_FULL = {
    "nv": "melanocytic nevi (benign mole)",
    "mel": "melanoma (dangerous)",
    "bkl": "benign keratosis",
    "bcc": "basal cell carcinoma",
    "akiec": "actinic keratoses",
    "vasc": "vascular lesion",
    "df": "dermatofibroma",
}
class_names = sorted(meta["dx"].unique())
n_classes = len(class_names)
class_to_idx = {c: i for i, c in enumerate(class_names)}
print("\nclasses:", class_names)

In [ ]:
# build a full file path for every image_id (images live in part_1 or part_2)
img_dirs = [os.path.join(BASE, "HAM10000_images_part_1"),
            os.path.join(BASE, "HAM10000_images_part_2")]

def find_path(image_id):
    for d in img_dirs:
        p = os.path.join(d, image_id + ".jpg")
        if os.path.exists(p):
            return p
    return None

meta["path"] = meta["image_id"].apply(find_path)
meta = meta.dropna(subset=["path"]).reset_index(drop=True)
meta["label"] = meta["dx"].map(class_to_idx)
print("images found:", len(meta))

## 2. See the imbalance (the field lesson)

- A model that always guesses the biggest class already looks "good" on plain accuracy.
- That's why we'll judge with **macro-averaged** metrics and **per-class recall** — which treat rare-but-deadly classes (like melanoma) as seriously as common ones.

In [ ]:
counts = meta["dx"].value_counts()
plt.figure(figsize=(8, 4))
plt.bar(counts.index, counts.values, color="steelblue")
plt.ylabel("number of images"); plt.title("HAM10000 class imbalance")
for i, v in enumerate(counts.values):
    plt.text(i, v + 60, str(v), ha="center", fontsize=9)
plt.show()

biggest = counts.max() / counts.sum()
print(f"biggest class = {biggest:.0%} of the data")
print(f"-> a lazy model that always predicts '{counts.idxmax()}' scores ~{biggest:.0%} accuracy but is useless.")

In [ ]:
# look at one image per class
fig, axes = plt.subplots(1, n_classes, figsize=(15, 3))
for ax, c in zip(axes, class_names):
    row = meta[meta["dx"] == c].iloc[0]
    ax.imshow(keras.utils.load_img(row["path"], target_size=(100, 100)))
    ax.set_title(c, fontsize=10); ax.axis("off")
plt.suptitle("One example per lesion type")
plt.tight_layout(); plt.show()

## 3. Build tf.data pipelines from the file paths

- Split into train/test (stratified so every class appears in both).
- A small loader reads + resizes each image on demand — the `tf.data` version of a custom Dataset.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    meta, test_size=0.2, random_state=42, stratify=meta["label"])
print("train:", len(train_df), " test:", len(test_df))

IMG_SIZE = (224, 224)
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label   # NOTE: raw 0-255; each model's preprocess_input is inside the model

def make_ds(df, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((df["path"].values, df["label"].values))
    if shuffle:
        ds = ds.shuffle(len(df), seed=42)
    return ds.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(train_df, shuffle=True)
test_ds  = make_ds(test_df,  shuffle=False)
y_test = test_df["label"].values

## 4. The comparison: 5 architectures, same recipe

- For each backbone: load pretrained (ImageNet), **freeze**, add its own `preprocess_input` + pooling + a Dense head.
- Train **only the head** for a few epochs, then record: **accuracy, macro-F1, training time, params.**
- We build each model with a small factory so the comparison is fair.

In [ ]:
# each entry: constructor + its matching preprocess_input
ARCHS = {
    "InceptionV3":    (keras.applications.InceptionV3,    keras.applications.inception_v3.preprocess_input),
    "ResNet50":       (keras.applications.ResNet50,       keras.applications.resnet50.preprocess_input),
    "DenseNet121":    (keras.applications.DenseNet121,    keras.applications.densenet.preprocess_input),
    "MobileNetV2":    (keras.applications.MobileNetV2,    keras.applications.mobilenet_v2.preprocess_input),
    "EfficientNetB0": (keras.applications.EfficientNetB0, keras.applications.efficientnet.preprocess_input),
}

def build_model(ctor, preprocess):
    base = ctor(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    base.trainable = False                       # feature extraction
    inputs = keras.Input((224, 224, 3))
    x = preprocess(inputs)                       # model-specific scaling
    x = base(x, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(n_classes, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model, base

In [ ]:
from sklearn.metrics import f1_score

EPOCHS = 4        # a few epochs is enough to compare (raise for better final numbers)
results = []

for name, (ctor, prep) in ARCHS.items():
    print(f"\n=== {name} ===")
    model, base = build_model(ctor, prep)

    t0 = time.time()
    model.fit(train_ds, epochs=EPOCHS, verbose=0)
    train_time = time.time() - t0

    # predictions + metrics
    t1 = time.time()
    probs = model.predict(test_ds, verbose=0)
    infer_time = time.time() - t1
    preds = probs.argmax(1)

    acc = (preds == y_test).mean()
    macro_f1 = f1_score(y_test, preds, average="macro")   # imbalance-aware
    params = model.count_params()

    results.append({"model": name, "accuracy": acc, "macro_f1": macro_f1,
                    "train_s": train_time, "infer_s": infer_time, "params": params})
    print(f"acc {acc:.2%} | macro-F1 {macro_f1:.3f} | train {train_time:.0f}s | params {params/1e6:.1f}M")

## 5. The results table

In [ ]:
res = pd.DataFrame(results).sort_values("macro_f1", ascending=False).reset_index(drop=True)
res_display = res.copy()
res_display["accuracy"] = (res_display["accuracy"] * 100).round(1).astype(str) + "%"
res_display["macro_f1"] = res_display["macro_f1"].round(3)
res_display["train_s"] = res_display["train_s"].round(0).astype(int)
res_display["infer_s"] = res_display["infer_s"].round(1)
res_display["params"] = (res_display["params"] / 1e6).round(1).astype(str) + "M"
res_display.columns = ["Model", "Accuracy", "Macro-F1", "Train (s)", "Infer (s)", "Params"]
print(res_display.to_string(index=False))

## 6. Visualize the trade-offs

- **Accuracy vs Macro-F1** — note where they disagree (imbalance!).
- **Accuracy vs size** — the efficiency story from the CNN-history talk, made real.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# accuracy vs macro-F1
xr = np.arange(len(res))
ax1.bar(xr - 0.2, res["accuracy"] * 100, 0.4, label="accuracy", color="steelblue")
ax1.bar(xr + 0.2, res["macro_f1"] * 100, 0.4, label="macro-F1", color="coral")
ax1.set_xticks(xr); ax1.set_xticklabels(res["model"], rotation=30, ha="right")
ax1.set_ylabel("score (%)"); ax1.set_title("Accuracy vs Macro-F1"); ax1.legend(); ax1.grid(alpha=0.3)

# accuracy vs params (size)
ax2.scatter(res["params"] / 1e6, res["macro_f1"] * 100, s=120, color="green")
for _, r in res.iterrows():
    ax2.annotate(r["model"], (r["params"] / 1e6, r["macro_f1"] * 100),
                 textcoords="offset points", xytext=(6, 4), fontsize=9)
ax2.set_xlabel("parameters (millions)"); ax2.set_ylabel("macro-F1 (%)")
ax2.set_title("Performance vs model size"); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**How to read this (the field skill):**
- The most *accurate* model isn't always the right choice — a phone app wants **MobileNet** (tiny, fast), a hospital server might want the highest **macro-F1** regardless of size.
- Where **accuracy is high but macro-F1 is low**, the model is riding the imbalance — good at common lesions, bad at rare ones. On a cancer task, that's dangerous.
- This trade-off *is* the job. There's no single "best" — only best-for-a-purpose.

## 7. Fine-tune the winner

- Pick the top model by macro-F1.
- **Unfreeze** the backbone and train a bit more with a **very small learning rate** — gently adapting ImageNet features to skin lesions.
- This is the extra step that squeezes out the final accuracy.

In [ ]:
winner = res.iloc[0]["model"]
print("winner by macro-F1:", winner)

ctor, prep = ARCHS[winner]
model, base = build_model(ctor, prep)
model.fit(train_ds, epochs=4, verbose=0)          # train head first (as before)

# --- fine-tune: unfreeze backbone, tiny learning rate ---
base.trainable = True
model.compile(optimizer=keras.optimizers.Adam(1e-5),   # 100x smaller LR!
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, epochs=3, verbose=1)

probs = model.predict(test_ds, verbose=0)
preds = probs.argmax(1)
from sklearn.metrics import f1_score
print(f"\n{winner} after fine-tuning: acc {(preds==y_test).mean():.2%} | macro-F1 {f1_score(y_test,preds,average='macro'):.3f}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

cm = confusion_matrix(y_test, preds)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
plt.title(f"{winner} (fine-tuned) — skin lesion classification")
plt.tight_layout(); plt.show()

print(classification_report(y_test, preds, target_names=class_names, digits=3))

- Read the **per-class recall** in the report — especially for **`mel` (melanoma)**. Missing a melanoma (low recall) is the costly error in real dermatology.
- The confusion matrix shows *which* lesions get mixed up — often the visually similar ones, just like a human dermatologist's hard cases.

## Your turn (solo task) ✍️

Pick at least two:
1. **Handle the imbalance:** pass `class_weight` to `model.fit` (weight rare classes higher) and see if macro-F1 improves.
2. **Fine-tune a different model** from the table and compare to the winner.
3. **Train longer** (more epochs) for the top 2 models — does the ranking change?
4. **Add data augmentation** (random flip/rotation) and check the effect on macro-F1.

In [ ]:
# ===== YOUR EXPERIMENTS HERE =====
# tip for #1:
# from sklearn.utils.class_weight import compute_class_weight
# w = compute_class_weight('balanced', classes=np.arange(n_classes), y=train_df['label'].values)
# class_weight = {i: w[i] for i in range(n_classes)}
# ... model.fit(train_ds, epochs=..., class_weight=class_weight)


## Summary

- **Real task, real data:** classifying 7 skin-lesion types from HAM10000 — loaded CSV-driven, the way field datasets actually come.
- **Imbalance matters:** plain accuracy is misleading on a 67%-one-class dataset; we judged with **macro-F1** and **per-class recall**.
- **Architecture comparison:** InceptionV3, ResNet50, DenseNet121, MobileNetV2, EfficientNetB0 — traded off on **accuracy, speed, and size**. No single winner: MobileNet is tiny and fast, the bigger nets may score higher.
- **Fine-tuning the winner** (unfreeze + tiny LR) gave a final boost.
- The real skill isn't picking "the best model" — it's picking the **right model for the constraints** (device, latency, and the cost of each error).

**Tomorrow:** object detection — not just *what* is in an image, but *where*.

---
*A PyTorch version of this notebook is provided separately for reference — same comparison, same dataset, using `torchvision.models`.*